# 블록 1-2 · RAG 시스템 구성 요소 및 파이프라인 구축

- RAG 를 **파이프라인**으로 이해하고, Indexing 4단계와 Retrieval 4단계를 설명한다
- RAG 답변이 틀렸을 때 **어느 단계가 실패했는지 진단**하는 순서를 갖는다
- 검색 옵션(similarity / MMR / 임계값)과 하이브리드 검색의 **선택 기준**을 안다

---

## 0. 실행 준비

이 노트북은 `.env` 의 `OPENAI_API_KEY` 를 사용합니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), (
    "OPENAI_API_KEY 가 없습니다. 프로젝트 루트에 .env 를 만들고 키를 넣으세요."
)
print("준비 완료")

---

## 1. Indexing — Load → Split → Embed → Store

문서는 검색 이전에 4단계(Load → Split → Embed → Store)를 거쳐 인덱싱됩니다. 인덱싱 단계에서 원문 정보가 누락되거나 청크가 잘못 분할되면, 후속 검색 및 생성 단계에서 이를 복구할 수 없으므로 전처리 품질이 핵심입니다.

In [ ]:
from pathlib import Path
from langchain_core.documents import Document

# 실행 위치(프로젝트 루트 또는 하위 디렉터리)에 관계없이 데이터 디렉터리를 동적으로 탐색
DOC_DIR = next(p / "data" / "company_docs"
               for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "data" / "company_docs").is_dir())

# Load — 텍스트/마크다운 문서를 로드합니다. 실무에서 PDF나 복잡한 표를 로드할 때 파싱 오류가 자주 발생하므로 적절한 문서 로더 선택이 중요합니다.
docs = [Document(page_content=p.read_text(encoding="utf-8"), metadata={"source": p.name})
        for p in sorted(DOC_DIR.iterdir())]

for d in docs:
    print(f"{d.metadata['source']:<24} {len(d.page_content):>6,}자")
print(f"\n총 {len(docs)}개 문서 · {sum(len(d.page_content) for d in docs):,}자")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 구분자(SEPARATORS) 순서: 문서 계층 구조를 보존하기 위한 우선순위 설정
# 헤더(#, ##)를 최우선 구분자로 지정하여 섹션 단위의 의미적 응집성을 유지합니다.
SEPARATORS = ["\n## ", "\n### ", "\n\n", "\n", " "]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=80, separators=SEPARATORS,
)
chunks = splitter.split_documents(docs)

lens = [len(c.page_content) for c in chunks]
print(f"청크 {len(chunks)}개 · 평균 {sum(lens)//len(lens)}자 · 최소 {min(lens)} · 최대 {max(lens)}")
print("\n--- 3번째 청크 ---")
print(chunks[2].page_content[:220])

In [ ]:
from pathlib import Path
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

EMBED = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBED)

# 로컬 벡터 DB 저장 경로 설정 (중복 임베딩 방지 및 캐싱)
CHROMA_DIR = "./.chroma_db"

def get_or_create_vectorstore(chunks, collection_name: str, force_reload: bool = False) -> Chroma:
    """로컬 저장소에 인덱싱된 데이터가 있으면 로드하고, 없거나 force_reload=True인 경우 새로 인덱싱합니다."""
    if force_reload:
        Chroma(collection_name=collection_name, embedding_function=embeddings, persist_directory=CHROMA_DIR).delete_collection()

    vectorstore = Chroma(
        collection_name=collection_name,
        embedding_function=embeddings,
        persist_directory=CHROMA_DIR,
    )

    if vectorstore._collection.count() == 0:
        vectorstore.add_documents(chunks)
        print(f"[{collection_name}] 신규 인덱싱 완료 — 청크 {len(chunks)}개를 {EMBED}(1536차원)로 임베딩")
    else:
        print(f"[{collection_name}] 로컬 캐시에서 로드 완료 — 청크 {vectorstore._collection.count()}개 (API 호출 없음)")

    return vectorstore


vectorstore = get_or_create_vectorstore(chunks, "p1_02_main_500")
print("컬렉션 문서 수:", vectorstore._collection.count(), "← 재실행 시에도 멱등하게 유지됨")

---

## 2. Retrieval — 질의 임베딩 → 검색 → 주입 → 생성

In [ ]:
QUESTION = "재택근무는 주 몇 회까지 가능한가요?"

print("── similarity (기본) ──")
for i, d in enumerate(vectorstore.similarity_search(QUESTION, k=4), 1):
    print(f"{i}. [{d.metadata['source']}] {d.page_content[:44]}".replace("\n", " "))

# MMR 은 이미 뽑은 것과 겹치는 후보에 벌점을 준다. lambda_mult 가 작을수록 다양성 쪽.
print("\n── mmr (다양성 고려) ──")
for i, d in enumerate(
        vectorstore.max_marginal_relevance_search(QUESTION, k=4, fetch_k=20, lambda_mult=0.5), 1):
    print(f"{i}. [{d.metadata['source']}] {d.page_content[:44]}".replace("\n", " "))

In [ ]:
print("── relevance score 분포 ──")
for d, s in vectorstore.similarity_search_with_relevance_scores(QUESTION, k=5):
    print(f"  {s:.4f}  [{d.metadata['source']:<20}] {d.page_content[:34]}".replace("\n", " "))

# 흔히 쓰는 "0.7 이상만" 규칙을 그대로 걸어 본다.
strict = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 5, "score_threshold": 0.7},
)
print(f"\n임계값 0.7 → 통과한 문서: {len(strict.invoke(QUESTION))}건")

loose = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 5, "score_threshold": 0.5},
)
print(f"임계값 0.5 → 통과한 문서: {len(loose.invoke(QUESTION))}건")

---

### 📊 결과 해석

**예상되는 결과:** 1위 문서의 relevance score 가 0.32 수준이고, 임계값 0.7 은 물론 0.5 로도 통과 0건. 정답을 제대로 찾았는데도 전부 버려진다.

| 결과 | 해설 및 원인 분석 |
|---|---|
| 예상대로 나옴 | 이 규칙을 그대로 배포했으면 **모든 질문에 '모르겠습니다'** 라고 답하는 봇이 됐을 겁니다. |
| 반대로 나옴 | 점수가 높게 나왔다면 임베딩·거리 설정이 다른 것입니다. 그게 바로 요점입니다 — "같은 0.7 이 환경에 따라 다른 뜻"이라는 사실이 더 강하게 증명됩니다. |
| 차이가 없음 | 두 임계값 결과가 같아도 상관없습니다. **점수의 절대값에 의미가 없다**는 쪽으로 확인합니다. |

---

## 3. BM25 와 하이브리드 검색

In [ ]:
import re
from rank_bm25 import BM25Okapi

# 영문/숫자 토큰은 그대로, 한글은 조사 문제를 피하려 2-gram 을 함께 넣는다.
# (형태소 분석기 없이 쓰는 교육용 근사 — 실무에서는 Kiwi/Mecab 을 쓴다)
TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z0-9._/@-]*|[0-9][0-9,.\-]*|[가-힣]+")


def tokenize(text: str) -> list[str]:
    out = []
    for t in TOKEN_RE.findall(text.lower()):
        out.append(t)
        if re.fullmatch(r"[가-힣]+", t) and len(t) > 2:
            out += [t[i:i + 2] for i in range(len(t) - 1)]
    return out


bm25 = BM25Okapi([tokenize(c.page_content) for c in chunks])


def bm25_search(query: str, k: int = 4):
    scores = bm25.get_scores(tokenize(query))
    order = sorted(range(len(chunks)), key=lambda i: -scores[i])[:k]
    return [chunks[i] for i in order]


print("BM25 준비 완료 —", len(chunks), "청크")
print("샘플 토큰:", tokenize("내선 9020번은 어느 팀 번호인가요?"))

In [ ]:
def rrf_search(query: str, k: int = 4, pool: int = 10, const: int = 60):
    """두 검색기의 '순위'만 융합한다 — 점수는 쓰지 않는다."""
    content_to_idx = {c.page_content: i for i, c in enumerate(chunks)}
    fused: dict[int, float] = {}

    for rank, d in enumerate(vectorstore.similarity_search(query, k=pool)):
        i = content_to_idx.get(d.page_content)
        if i is not None:
            fused[i] = fused.get(i, 0.0) + 1.0 / (const + rank + 1)

    scores = bm25.get_scores(tokenize(query))
    for rank, i in enumerate(sorted(range(len(chunks)), key=lambda i: -scores[i])[:pool]):
        fused[i] = fused.get(i, 0.0) + 1.0 / (const + rank + 1)

    return [chunks[i] for i in sorted(fused, key=lambda i: -fused[i])[:k]]


print("RRF 준비 완료")

---

## 4. RAG 완주와 검색 실패 진단

### 먼저 파이프라인을 완주합니다

검색 결과를 프롬프트에 넣어 답을 만드는, 가장 단순한 RAG 체인입니다.
`문맥에 없으면 모른다고 하라`는 지시가 들어 있는 것에 주목하세요 —
이게 없으면 ⑤미기권이 바로 발생합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 사내 규정 안내 담당자입니다. 아래 [문맥]에 있는 내용만 근거로 답하세요.\n"
     "문맥에 답이 없으면 반드시 '문서에서 찾지 못했습니다'라고 답하세요. 지어내지 마세요."),
    ("human", "[문맥]\n{context}\n\n[질문] {question}"),
])


def answer(question: str, retrieved: list) -> str:
    """검색을 체인 밖에 두는 이유: 어떤 청크가 들어갔는지 눈으로 봐야 진단이 된다."""
    context = "\n\n---\n\n".join(d.page_content for d in retrieved)
    chain = RAG_PROMPT | llm
    return chain.invoke({"context": context, "question": question}).content


hits = vectorstore.similarity_search(QUESTION, k=4)
print("[질문]", QUESTION)
print("[답변]", answer(QUESTION, hits))
print("\n[근거 출처]", [d.metadata["source"] for d in hits])

정상 동작을 확인했습니다. 여기까지가 "RAG 완주"입니다.
이제 **일부러 실패하는 질의**를 넣습니다.

---

## 🔮 예측 — D3

사내 문서에는 재무팀 내선번호가 **9020** 이라고 분명히 적혀 있습니다.
`"내선 9020번은 어느 팀 번호인가요?"` 라고 물으면 어떻게 될까요?

① 정확히 답한다 ② 다른 팀 번호를 답한다(환각) ③ 모른다고 답한다**


<details>
<summary>생각해보기</summary>

> **이 질문의 의도:** "문서에 있는데 왜 못 찾지?"를 겪게 하려는 것입니다. 그리고 그 실패가 **생성이 아니라 검색**에서 났다는 것을 눈으로 확인시킵니다.

</details>


In [ ]:
FAIL_Q = "내선 9020번은 어느 팀 번호인가요?"

# 먼저 답부터 본다. 진단은 그다음이다 — 현장 순서와 같게 간다.
vec_hits = vectorstore.similarity_search(FAIL_Q, k=4)
print("[질문]", FAIL_Q)
print("[답변]", answer(FAIL_Q, vec_hits))

### 🔍 실패 원인 진단 — 프롬프트 수정 전 검색 결과 확인

답변이 잘못되었을 때 프롬프트를 먼저 수정하기 쉽지만, **반드시 검색 단계의 결과부터 확인**해야 합니다.
RAG 파이프라인 구조상 생성 단계(LLM)는 검색기가 전달한 컨텍스트 내에서만 작동하므로, 검색 단계에서 올바른 근거를 가져오지 못하면 프롬프트를 아무리 개선해도 정답을 도출할 수 없습니다.

In [ ]:
print("── 벡터 검색이 실제로 가져온 4개 ──")
for i, d in enumerate(vec_hits, 1):
    print(f"{i}. [{d.metadata['source']}] {d.page_content[:60]}".replace("\n", " "))

# 검색 평가: 정답 근거 청크가 검색 결과 상위에 포함되었는지 확인
found = any("9020" in d.page_content for d in vec_hits)
print(f"\n★ 근거('9020' 포함 청크)가 검색 결과에 있는가? → {found}")

# 코퍼스 내 실제 정답 청크 검색 및 원문 확인
target_chunks = [c for c in chunks if "9020" in c.page_content]
print(f"★ 코퍼스 전체에는 존재하는가? → {bool(target_chunks)}")

if target_chunks:
    print("\n── 코퍼스 내 실제 정답 청크 원문 ──")
    for c in target_chunks:
        print(f"[{c.metadata.get('source', '')}]\n{c.page_content.strip()}")

> 💡 **진단 결과: 검색 실패 (Retrieval Failure)**
>
> - **원인 분석:** 원본 문서에는 정답이 존재하지만 검색기가 해당 청크를 가져오지 못했습니다. (환각이나 미기권이 아니므로 프롬프트 수정으로는 해결 불가)
> - **임베딩 검색의 한계:** 코퍼스 내에 여러 내선번호(`1234`, `5678`, `9020` 등)가 존재할 때, 밀집 임베딩 모델은 이들을 모두 '연락처 안내 문장'이라는 유사한 의미 공간으로 임베딩합니다. 따라서 의미적 유사성은 높게 평가하지만, 특정 고유 번호(`9020`)와 같은 정밀한 키워드 구분에는 취약합니다.
> - **해결 방향:** 키워드 정확도가 높은 **BM25** 또는 **하이브리드(RRF) 검색**을 적용해야 합니다.

**대조 실행** — 아래 두 셀을 연달아 실행합니다.

- **A:** BM25 — 드문 토큰 `9020` 에 IDF 가 크게 걸린다
- **B:** RRF — 벡터와 BM25 의 순위를 융합한다

In [ ]:
print("── BM25 단독 ──")
bm_hits = bm25_search(FAIL_Q, k=4)
for i, d in enumerate(bm_hits, 1):
    mark = "★" if "9020" in d.page_content else " "
    print(f"{mark}{i}. [{d.metadata['source']}] {d.page_content[:56]}".replace("\n", " "))
print("근거 포함:", any("9020" in d.page_content for d in bm_hits))

In [ ]:
print("── 하이브리드(RRF) ──")
rrf_hits = rrf_search(FAIL_Q, k=4)
for i, d in enumerate(rrf_hits, 1):
    mark = "★" if "9020" in d.page_content else " "
    print(f"{mark}{i}. [{d.metadata['source']}] {d.page_content[:56]}".replace("\n", " "))

print("\n[답변 — 하이브리드]", answer(FAIL_Q, rrf_hits))

> 💡 **핵심 고찰 — RRF(하이브리드) 순위 산정 메커니즘과 한계**
>
> - **순위 변화 분석:** 정답 청크가 BM25에서는 **1위**였으나 RRF 종합 순위에서는 **4위**로 내려왔습니다. 이는 오류가 아닌 RRF 알고리즘의 동작 특성 때문입니다. 해당 청크가 벡터 검색 풀(Top-10)에 포함되지 못해 RRF 점수를 1회만 합산받은 반면, 양쪽 검색기에서 고르게 중상위권을 차지한 문서들이 더 높은 종합 점수를 얻었기 때문입니다.
> - **하이브리드 검색의 한계:** 한쪽 검색기에서만 강하게 반응하는 문서는 반환 청크 수(`k`)를 좁힐 경우(예: `k=2`) 최종 컨텍스트에서 누락될 위험이 있습니다.
> - **리랭커(Reranker)의 필요성:** 이러한 순위 융합의 한계를 보완하기 위해 1단계에서 하이브리드로 후보군을 넓게 수집하고, 2단계에서 **Cross-Encoder 기반 리랭커**로 문맥 연관성을 정밀 재평가하는 2단계 파이프라인이 실무에서 널리 활용됩니다.

> ### ▶ 함께 실행 — D3
>
> **정상 질의와 실패 질의를 차례로 넣어 봅니다.**
>
> 위 셀을 여러분 노트북에서도 실행해 보세요.
>
> 🔍 **여러분 화면에서 볼 것** — 검색된 청크 안에 답이 있는가 — **생성이 아니라 검색이 실패**했음을 눈으로 확인하세요
>
> ⚠️ 실행 결과가 예시와 **다를 수 있습니다.** 이는 오류가 아니며, 생성 모델의 통계적 변동성을 확인하는 과정입니다.

---

### 📊 결과 해석

**예상되는 결과:** 벡터 검색 단독으로는 정답 청크를 상위권에 가져오지 못하지만, BM25(1위) 및 RRF 하이브리드(상위 4위 이내)로 순위를 끌어올려 최종적으로 '재무팀'이라는 정확한 답변을 생성합니다.

| 결과 상황 | 해설 및 원인 분석 |
|---|---|
| 예상대로 출력됨 (하이브리드 성공) | 프롬프트 수정 없이 **검색 방식(BM25/RRF)만 개선하여 정답을 도출**했습니다. 이는 RAG 문제 해결 시 '진단은 역방향(생성 → 검색), 개선은 정방향(검색 → 생성)' 원칙의 대표적인 사례입니다. |
| 벡터 검색도 성공한 경우 | 임베딩 모델 및 청크 분할 설정에 따라 벡터 검색에서도 정답 청크가 상위에 잡힐 수 있습니다. 고유 식별자나 특수 키워드(예: 모델명, 코드값) 질의를 통해 키워드 검색과 벡터 검색의 성능 차이를 추가로 비교해 보세요. |
| 검색 결과 차이가 모호한 경우 | 반환 청크 수(`k=2`)를 좁혀 설정하면 상위 랭킹 정밀도 차이를 더욱 명확하게 관찰할 수 있습니다. |

### 💡 실무 RAG 인사이트 — 환각(Hallucination)보다 감지하기 어려운 '기권율'

단일 벡터 검색 환경에서 모델은 임의로 허위 사실을 지어내는 환각을 일으키기보다, **3회 모두 정직하게 기권(근거 부족, "문서에서 찾지 못했습니다")** 하는 동작을 보였습니다. 프롬프트 제약을 다소 완화하더라도 인접한 번호(`9012`, 인사팀)를 왜곡하여 연결하지 않았습니다.

이러한 현상이 실무 운영에서 특히 중요한 이유는 **기권 실패의 비가시성(Invisibility)** 때문입니다.

- **환각(오답 생성)**: 잘못된 정보가 명시적으로 노출되므로 사용자 피드백이나 오류 보고를 통해 즉각 인지됩니다.
- **기권(답변 포기)**: 사용자가 "관련 문서가 원래 없나 보다" 하고 넘어가기 쉬워, 검색 로그를 능동적으로 분석하지 않으면 시스템 결함이 장기간 방치됩니다.

따라서 사내 RAG 시스템 평가 및 모니터링 시에는 **단순 생성 정확도뿐만 아니라 '기권율(Abstention Rate)'과 검색 누락 여부를 반드시 함께 측정**해야 합니다.

> 🎯 **핵심** — RAG 답변이 틀렸을 때 대부분의 원인은 **검색**이다. 프롬프트를 고치기 전에 **검색된 청크를 열어라.** 근거가 안 들어왔다면 그건 생성의 문제가 아니다.

| 흔한 오해 | 실제 |
|---|---|
| RAG 답변이 틀리면 LLM 이 나쁜 것이다 | 5유형 중 프롬프트로 풀리는 건 2개(환각·미기권)뿐. 나머지는 파이프라인 앞단 |

---

## 5. 청크 사이즈 — 청크 300 vs 1200

## 🔮 예측 — D4

**같은 문서를 청크 **300** 과 **1200** 으로 각각 인덱싱합니다.
**어느 쪽이 더 나은 검색 결과를 줄까요?** ① 300 ② 1200 ③ 상황에 따라 다르다**


<details>
<summary>생각해보기</summary>

> **이 질문의 의도:** ③을 고르는 분이 많습니다. 그런데 "그래서 어느 상황에서?"를 물으면 대개 답이 없습니다. 여기서는 **뒤집히는 지점을 정확히** 봅니다.

</details>


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 300자 및 1200자 크기로 각각 분할 및 벡터스토어 생성
splitter_300 = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=50, separators=SEPARATORS,
)
splitter_1200 = RecursiveCharacterTextSplitter(
    chunk_size=1200, chunk_overlap=150, separators=SEPARATORS,
)

chunks_300 = splitter_300.split_documents(docs)
chunks_1200 = splitter_1200.split_documents(docs)

vs_small = get_or_create_vectorstore(chunks_300, "p1_02_k300")
vs_large = get_or_create_vectorstore(chunks_1200, "p1_02_k1200")

print(f"vs_small (chunk=300) : 청크 {len(chunks_300)}개")
print(f"vs_large (chunk=1200): 청크 {len(chunks_1200)}개")

**대조 실행** — 아래 두 셀을 연달아 실행합니다.

- **A:** Q-A 단답형 — "야근 식대는 얼마까지 지원되나요?" (정답: 1만 원)
- **B:** Q-B 표 조회형 — "동남아 출장 일비는 얼마인가요?" (정답: $50/일)

In [ ]:
Q_A = "야근 식대는 얼마까지 지원되나요?"
Q_B = "동남아 출장 일비는 얼마인가요?"
GOLD = {Q_A: "1만 원", Q_B: "$50"}


def probe(vs, question, k=3):
    hits = vs.similarity_search(question, k=k)
    ctx = "\n\n".join(d.page_content for d in hits)
    return ("O" if GOLD[question] in ctx else "X"), len(ctx), hits


print(f"{'질문':<28}{'chunk=300':>14}{'chunk=1200':>14}")
for q in (Q_A, Q_B):
    s_ok, s_len, _ = probe(vs_small, q)
    l_ok, l_len, _ = probe(vs_large, q)
    print(f"{q[:26]:<28}{s_ok + f' ({s_len:>4}자)':>14}{l_ok + f' ({l_len:>4}자)':>14}")

In [ ]:
# 청크 크기(300자 vs 1200자)에 따른 최종 생성 답변 비교
for q in (Q_A, Q_B):
    for label, vs in (("300 ", vs_small), ("1200", vs_large)):
        _, _, hits = probe(vs, q)
        print(f"[{label}] {q}\n       → {answer(q, hits)[:80]}")
    print()

> ### ▶ 함께 실행 — D4
>
> **청크 300 / 1200 두 인덱스에 같은 질문을 넣습니다.**
>
> 위 셀을 여러분 노트북에서도 실행해 보세요.
>
> 🔍 **여러분 화면에서 볼 것** — 어느 쪽이 이기는가. **질문 유형에 따라 승자가 뒤바뀝니다**
>
> ⚠️ 실행 결과가 예시와 **다를 수 있습니다.** 이는 오류가 아니며, 생성 모델의 통계적 변동성을 확인하는 과정입니다.

---

### 📊 결과 해석

**예상되는 결과:** **승자가 질문마다 뒤바뀐다.** 제작 실측(3회 반복 3/3 동일): Q-A 는 300 성공(710자)·1200 실패(3296자), Q-B 는 300 실패(545자)·1200 성공(3231자).

| 결과 | 해설 및 원인 분석 |
|---|---|
| 예상대로 나옴 | **같은 문서, 같은 모델, 같은 프롬프트입니다.** 청크 크기 하나만 바꿨는데 한쪽은 맞고 한쪽은 틀렸습니다. 그리고 어느 쪽도 항상 이기지 않았습니다. |
| 반대로 나옴 | 한쪽이 둘 다 이겼다면: "이 코퍼스에서는 그렇습니다. 그런데 **여러분 문서에서도 그럴까요?** 그걸 알 방법은 하나뿐입니다 — 오라클 테스트로 측정하는 것입니다." |
| 차이가 없음 | 차이가 없으면 문자 수를 보세요. 같은 정답을 얻는 데 1200 은 **4~6배의 컨텍스트**를 태웠습니다. 비용과 Lost in the Middle 이 거기서 발생합니다. |

### 왜 이렇게 뒤집혔는가 — 실패를 하나씩 열어보면

두 실패의 원인이 정반대입니다.

**Q-A 에서 1200 이 진 이유 — 노이즈 희석.**
"야근 식대 1만 원"은 경비 규정 §5.4 의 **두 줄짜리** 항목입니다.
1200자 청크에 넣으면 회의비·다과비·팀 회식비와 한 덩어리가 되고,
그 덩어리의 평균적 의미는 "회의비 규정"에 가까워집니다. 질의와의 거리가 멀어집니다.

**Q-B 에서 300 이 진 이유 — 문맥 소실.**
"동남아 $50/일"은 **표의 한 행**입니다. 300자로 자르면 `| 동남아 | $50/일 | - |` 만 남고,
이 표가 "해외 출장 일비" 표라는 정보가 **다른 청크로 떨어져 나갑니다.**
행만 남은 청크는 무엇에 대한 표인지 알 수 없으니 질의와 가까워지지 못합니다.

> 그래서 청킹의 정답은 크기가 아니라 **구조**입니다.
> 예를 들면, Parent-Child·Contextual Retrieval 이 정확히 이 두 실패를 동시에 노립니다 —
> **작게 검색하고 크게 생성하거나, 조각에 출신 정보를 붙여 두거나.**

그리고 실무적으로 더 중요한 결론은 이것입니다.

> **"우리 회사에 맞는 청크 크기"는 고민해서 정하는 값이 아니라 직접 측정해서 정하는 값입니다.**
> 따라서, 오라클 테스트 20건이 있어야 합니다. 

---

## 6. 미니실습 M2 — top-k 를 줄여 본다

---

### 🔧 미니실습 M2 — 검색 개수 k 를 4 에서 1 로 바꾼다

**바꿀 것**: `K = 4` 의 숫자 하나

**볼 것**: ① **청크 개수와 컨텍스트 글자 수**(결정론 — 전원 같습니다) ② 답이 여전히 맞는가(관찰 항목)

> 3~5분 드립니다. 안 되면 **바로 아래 정답 셀을 실행**하고 따라오세요 — 여기서 막혀도 다음 내용에는 영향이 없습니다.

In [ ]:
# TODO — K 를 4 에서 1 로 바꿔 보세요.
K = 4

hits = vectorstore.similarity_search(QUESTION, k=K)
ctx = "\n\n".join(d.page_content for d in hits)
print(f"k={K} · 청크 {len(hits)}개 · 컨텍스트 {len(ctx):,}자")
print("-" * 52)
print(ctx[:300], "...")

<details>
<summary>정답 — M2</summary>

아래 셀이 기준 구현입니다. 직접 푼 결과와 비교해 보세요.

</details>

In [ ]:
for K in (4, 1):
    hits = vectorstore.similarity_search(QUESTION, k=K)
    ctx = "\n\n".join(d.page_content for d in hits)
    has_answer = "주 2회" in ctx or "2회" in ctx
    print(f"k={K} · 청크 {len(hits)}개 · 컨텍스트 {len(ctx):>6,}자 · "
          f"근거 포함 {'✅' if has_answer else '❌'}")

> 🎯 **핵심** — `k` 는 **비용과 재현율을 맞바꾸는 손잡이**입니다.
> 크게 하면 답이 들어올 확률이 오르지만 *Lost in the Middle* 로 **묻히고**, 비용이 오릅니다.
> 정석은 k 를 키우는 것이 아니라 **넓게 뽑고 리랭킹으로 정제**하는 것입니다.

---

## 정리

- RAG는 단순한 모델이 아니라 **엔드투엔드 파이프라인**입니다. 전처리 및 검색 단계에서 손실된 정보는 생성 단계에서 복구할 수 없습니다
- **진단은 역방향, 개선은 정방향.** 프롬프트를 만지기 전에 검색된 청크를 연다
- 임베딩은 의미가 아니라 공기(共起) 경향을 잰다 — 그래서 번호·코드에 약하고, 그 자리를 **BM25 + RRF** 가 메운다
- 점수 임계값·청크 크기·top-k 는 **정하는 값이 아니라 측정하는 값**이다. 그래서 오라클 테스트 20건이 첫 산출물이다